In [2]:
from pathlib import Path
import pandas as pd
import json

# Change this if your CSVs are stored elsewhere.
DATA_DIR = Path("../data/ton_iot")

# The notebook expects one CSV per IoT device.
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Data folder: {DATA_DIR.resolve()}")
print(f"CSV files found: {len(csv_files)}")
for path in csv_files:
    print(f" - {path.name}")

Data folder: C:\Aayush\ton-iot-federated-IDS\data\ton_iot
CSV files found: 7
 - Train_Test_IoT_Fridge.csv
 - Train_Test_IoT_Garage_Door.csv
 - Train_Test_IoT_GPS_Tracker.csv
 - Train_Test_IoT_Modbus.csv
 - Train_Test_IoT_Motion_Light.csv
 - Train_Test_IoT_Thermostat.csv
 - Train_Test_IoT_Weather.csv


In [3]:
file_info = []

for path in csv_files:
    # Read only the header first; this does not load the full dataset.
    columns = pd.read_csv(path, nrows=0).columns.tolist()

    file_info.append({
        "file": path.name,
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2),
        "column_count": len(columns),
        "columns": columns,
    })

pd.DataFrame(file_info)[["file", "size_mb", "column_count"]]

,file,size_mb,column_count
0,Train_Test_IoT_Fridge.csv,1.60,6
1,Train_Test_IoT_Garage_Door.csv,1.66,6
2,Train_Test_IoT_GPS_Tracker.csv,2.02,6
3,Train_Test_IoT_Modbus.csv,1.62,8
4,Train_Test_IoT_Motion_Light.csv,1.40,6
5,Train_Test_IoT_Thermostat.csv,1.40,6
6,Train_Test_IoT_Weather.csv,2.47,7


In [4]:
for item in file_info:
    print(f"\n{item['file']} ({item['column_count']} columns)")
    print(item["columns"])


Train_Test_IoT_Fridge.csv (6 columns)
['date', 'time', 'fridge_temperature', 'temp_condition', 'label', 'type']

Train_Test_IoT_Garage_Door.csv (6 columns)
['date', 'time', 'door_state', 'sphone_signal', 'label', 'type']

Train_Test_IoT_GPS_Tracker.csv (6 columns)
['date', 'time', 'latitude', 'longitude', 'label', 'type']

Train_Test_IoT_Modbus.csv (8 columns)
['date', 'time', 'FC1_Read_Input_Register', 'FC2_Read_Discrete_Value', 'FC3_Read_Holding_Register', 'FC4_Read_Coil', 'label', 'type']

Train_Test_IoT_Motion_Light.csv (6 columns)
['date', 'time', 'motion_status', 'light_status', 'label', 'type']

Train_Test_IoT_Thermostat.csv (6 columns)
['date', 'time', 'current_temperature', 'thermostat_status', 'label', 'type']

Train_Test_IoT_Weather.csv (7 columns)
['date', 'time', 'temperature', 'pressure', 'humidity', 'label', 'type']


In [5]:
datasets = {}

for path in csv_files:
    df = pd.read_csv(path, low_memory=False)
    datasets[path.stem] = df
    print(f"{path.name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

Train_Test_IoT_Fridge.csv: 39,944 rows × 6 columns
Train_Test_IoT_Garage_Door.csv: 39,587 rows × 6 columns
Train_Test_IoT_GPS_Tracker.csv: 38,960 rows × 6 columns
Train_Test_IoT_Modbus.csv: 31,106 rows × 8 columns
Train_Test_IoT_Motion_Light.csv: 39,488 rows × 6 columns
Train_Test_IoT_Thermostat.csv: 32,774 rows × 6 columns
Train_Test_IoT_Weather.csv: 39,260 rows × 7 columns


In [6]:
schemas = {name: set(df.columns) for name, df in datasets.items()}

all_columns = set.union(*schemas.values()) if schemas else set()
schema_rows = []

for name, df in datasets.items():
    schema_rows.append({
        "device_file": name,
        "column_count": len(df.columns),
        "columns_missing_vs_union": sorted(all_columns - set(df.columns)),
        "columns_extra_vs_others": sorted(set(df.columns) - set.intersection(*schemas.values())),
    })

pd.DataFrame(schema_rows)

,device_file,column_count,columns_missing_vs_union,columns_extra_vs_others
0,Train_Test_IoT_Fridge,6,"[FC1_Read_Input_Register, FC2_Read_Discrete_Va...","[fridge_temperature, temp_condition]"
1,Train_Test_IoT_Garage_Door,6,"[FC1_Read_Input_Register, FC2_Read_Discrete_Va...","[door_state, sphone_signal]"
2,Train_Test_IoT_GPS_Tracker,6,"[FC1_Read_Input_Register, FC2_Read_Discrete_Va...","[latitude, longitude]"
3,Train_Test_IoT_Modbus,8,"[current_temperature, door_state, fridge_tempe...","[FC1_Read_Input_Register, FC2_Read_Discrete_Va..."
4,Train_Test_IoT_Motion_Light,6,"[FC1_Read_Input_Register, FC2_Read_Discrete_Va...","[light_status, motion_status]"
5,Train_Test_IoT_Thermostat,6,"[FC1_Read_Input_Register, FC2_Read_Discrete_Va...","[current_temperature, thermostat_status]"
6,Train_Test_IoT_Weather,7,"[FC1_Read_Input_Register, FC2_Read_Discrete_Va...","[humidity, pressure, temperature]"


In [7]:
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Possible target columns:", [c for c in df.columns if c.lower() in {"label", "type", "attack", "attack_type"}])

    for target in ["label", "type"]:
        if target in df.columns:
            print(f"\n{target} value counts:")
            print(df[target].value_counts(dropna=False).head(30))


--- Train_Test_IoT_Fridge ---
Possible target columns: ['label', 'type']

label value counts:
label
1    24944
0    15000
Name: count, dtype: int64

type value counts:
type
normal        15000
ddos           5000
backdoor       5000
injection      5000
password       5000
ransomware     2902
xss            2042
Name: count, dtype: int64

--- Train_Test_IoT_Garage_Door ---
Possible target columns: ['label', 'type']

label value counts:
label
1    24587
0    15000
Name: count, dtype: int64

type value counts:
type
normal        15000
ddos           5000
backdoor       5000
injection      5000
password       5000
ransomware     2902
xss            1156
scanning        529
Name: count, dtype: int64

--- Train_Test_IoT_GPS_Tracker ---
Possible target columns: ['label', 'type']

label value counts:
label
1    23960
0    15000
Name: count, dtype: int64

type value counts:
type
normal        15000
ddos           5000
backdoor       5000
injection      5000
password       5000
ransomware     2

In [8]:
for name, df in datasets.items():
    print(f"\n--- {name} ---")

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print("Columns with missing values:")
    display(missing.to_frame("missing_count"))

    print("Data types:")
    display(df.dtypes.astype(str).to_frame("dtype"))


--- Train_Test_IoT_Fridge ---
Columns with missing values:


,missing_count


Data types:


,dtype
date,str
time,str
fridge_temperature,float64
temp_condition,str
label,int64
type,str



--- Train_Test_IoT_Garage_Door ---
Columns with missing values:


,missing_count


... (18 outputs left out)


--- Train_Test_IoT_Weather ---
Columns with missing values:


,missing_count


Data types:


,dtype
date,str
time,str
temperature,float64
pressure,float64
humidity,float64
label,int64
type,str


In [9]:
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print(f"Duplicate rows: {df.duplicated().sum():,}")

    for target in ["label", "type"]:
        if target in df.columns:
            print(f"\n{target} distribution:")
            display(
                df[target]
                .value_counts(dropna=False)
                .rename_axis(target)
                .reset_index(name="count")
            )


--- Train_Test_IoT_Fridge ---
Duplicate rows: 2,834

label distribution:


,label,count
0,1,24944
1,0,15000



type distribution:


,type,count
0,normal,15000
1,ddos,5000
2,backdoor,5000
3,injection,5000
4,password,5000
5,ransomware,2902
6,xss,2042



--- Train_Test_IoT_Garage_Door ---
Duplicate rows: 20,221

label distribution:


,label,count
0,1,24587
1,0,15000


... (18 outputs left out)


--- Train_Test_IoT_Weather ---
Duplicate rows: 0

label distribution:


,label,count
0,1,24260
1,0,15000



type distribution:


,type,count
0,normal,15000
1,ddos,5000
2,backdoor,5000
3,injection,5000
4,password,5000
5,ransomware,2865
6,xss,866
7,scanning,529


In [10]:
summary = []

for name, df in datasets.items():
    row = {
        "file": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
    }

    for target in ["label", "type"]:
        if target in df.columns:
            row[f"{target}_unique_values"] = int(df[target].nunique(dropna=False))

    summary.append(row)

audit_summary = pd.DataFrame(summary)
display(audit_summary)

output_path = Path("dataset_audit_summary.csv")
audit_summary.to_csv(output_path, index=False)
print(f"Saved summary to: {output_path.resolve()}")

,file,rows,columns,duplicate_rows,missing_cells,label_unique_values,type_unique_values
0,Train_Test_IoT_Fridge,39944,6,2834,0,2,7
1,Train_Test_IoT_Garage_Door,39587,6,20221,0,2,8
2,Train_Test_IoT_GPS_Tracker,38960,6,0,0,2,8
3,Train_Test_IoT_Modbus,31106,8,13314,0,2,6
4,Train_Test_IoT_Motion_Light,39488,6,13658,0,2,8
5,Train_Test_IoT_Thermostat,32774,6,424,0,2,7
6,Train_Test_IoT_Weather,39260,7,0,0,2,8


Saved summary to: C:\Aayush\ton-iot-federated-IDS\notebooks\dataset_audit_summary.csv


In [11]:
TARGET_COLUMNS = {"label", "type"}

for name, df in datasets.items():
    features = [c for c in df.columns if c.lower() not in TARGET_COLUMNS]
    print(f"\n{name} — {len(features)} input features")
    print(features)

# Exact feature-column intersection across all seven files
feature_sets = [
    {c for c in df.columns if c.lower() not in TARGET_COLUMNS}
    for df in datasets.values()
]

common_features = set.intersection(*feature_sets)
all_features = set.union(*feature_sets)

print("\nCommon input features:", sorted(common_features))
print("\nAll distinct input features:", sorted(all_features))


Train_Test_IoT_Fridge — 4 input features
['date', 'time', 'fridge_temperature', 'temp_condition']

Train_Test_IoT_Garage_Door — 4 input features
['date', 'time', 'door_state', 'sphone_signal']

Train_Test_IoT_GPS_Tracker — 4 input features
['date', 'time', 'latitude', 'longitude']

Train_Test_IoT_Modbus — 6 input features
['date', 'time', 'FC1_Read_Input_Register', 'FC2_Read_Discrete_Value', 'FC3_Read_Holding_Register', 'FC4_Read_Coil']

Train_Test_IoT_Motion_Light — 4 input features
['date', 'time', 'motion_status', 'light_status']

Train_Test_IoT_Thermostat — 4 input features
['date', 'time', 'current_temperature', 'thermostat_status']

Train_Test_IoT_Weather — 5 input features
['date', 'time', 'temperature', 'pressure', 'humidity']

Common input features: ['date', 'time']

All distinct input features: ['FC1_Read_Input_Register', 'FC2_Read_Discrete_Value', 'FC3_Read_Holding_Register', 'FC4_Read_Coil', 'current_temperature', 'date', 'door_state', 'fridge_temperature', 'humidity', 'la

In [12]:
for name, df in datasets.items():
    print(f"\n--- {name} ---")

    if "label" in df.columns and "type" in df.columns:
        print("label × type:")
        display(pd.crosstab(df["label"], df["type"], margins=True))

    feature_cols = [
        c for c in df.columns
        if c.lower() not in TARGET_COLUMNS
    ]

    display(df[feature_cols].dtypes.astype(str).to_frame("dtype"))
    display(df[feature_cols].head(3))


--- Train_Test_IoT_Fridge ---
label × type:


type,backdoor,ddos,injection,normal,password,ransomware,xss,All
label,,,,,,,,
0,0,0,0,15000,0,0,0,15000
1,5000,5000,5000,0,5000,2902,2042,24944
All,5000,5000,5000,15000,5000,2902,2042,39944


,dtype
date,str
time,str
fridge_temperature,float64
temp_condition,str


,date,time,fridge_temperature,temp_condition
0,25-Apr-19,19:19:40,9.00,high
1,25-Apr-19,19:19:40,9.25,high
2,25-Apr-19,19:19:45,12.65,high



--- Train_Test_IoT_Garage_Door ---
label × type:


type,backdoor,ddos,injection,normal,password,ransomware,scanning,xss,All
label,,,,,,,,,
0,0,0,0,15000,0,0,0,0,15000
1,5000,5000,5000,0,5000,2902,529,1156,24587
All,5000,5000,5000,15000,5000,2902,529,1156,39587


... (18 outputs left out)


--- Train_Test_IoT_Weather ---
label × type:


type,backdoor,ddos,injection,normal,password,ransomware,scanning,xss,All
label,,,,,,,,,
0,0,0,0,15000,0,0,0,0,15000
1,5000,5000,5000,0,5000,2865,529,866,24260
All,5000,5000,5000,15000,5000,2865,529,866,39260


,dtype
date,str
time,str
temperature,float64
pressure,float64
humidity,float64


,date,time,temperature,pressure,humidity
0,25-Apr-19,17:33:16,40.881866,-0.101806,38.363631
1,25-Apr-19,17:33:16,44.913806,2.661616,46.141423
2,25-Apr-19,17:33:16,38.295822,-2.438871,50.850643


In [13]:
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("date examples:", df["date"].head(5).tolist())
    print("time examples:", df["time"].head(5).tolist())

    feature_cols = [
        c for c in df.columns
        if c.lower() not in {"label", "type", "date", "time"}
    ]
    print("Device feature dtypes:")
    display(df[feature_cols].dtypes.astype(str).to_frame("dtype"))


--- Train_Test_IoT_Fridge ---
date examples: ['25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19']
time examples: [' 19:19:40 ', ' 19:19:40 ', ' 19:19:45 ', ' 19:19:45 ', ' 19:19:55 ']
Device feature dtypes:


,dtype
fridge_temperature,float64
temp_condition,str



--- Train_Test_IoT_Garage_Door ---
date examples: ['25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19']
time examples: [' 14:42:33 ', ' 14:42:38 ', ' 14:42:43 ', ' 14:42:48 ', ' 14:42:58 ']
Device feature dtypes:


,dtype
door_state,str
sphone_signal,str



--- Train_Test_IoT_GPS_Tracker ---
date examples: ['25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19']
time examples: [' 18:31:39 ', ' 18:31:41 ', ' 18:31:46 ', ' 18:31:49 ', ' 18:31:51 ']
Device feature dtypes:


,dtype
latitude,float64
longitude,float64


... (4 outputs left out)


--- Train_Test_IoT_Thermostat ---
date examples: ['25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19']
time examples: ['8:59:02', '8:59:06', '8:59:06', '8:59:06', '8:59:06']
Device feature dtypes:


,dtype
current_temperature,float64
thermostat_status,int64



--- Train_Test_IoT_Weather ---
date examples: ['25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19']
time examples: ['17:33:16', '17:33:16', '17:33:16', '17:33:21', '17:33:21']
Device feature dtypes:


,dtype
temperature,float64
pressure,float64
humidity,float64


In [14]:
for name, df in datasets.items():
    timestamps = pd.to_datetime(
        df["date"].astype(str) + " " + df["time"].astype(str),
        errors="coerce"
    )

    print(
        name,
        "| parse failures:", int(timestamps.isna().sum()),
        "| min:", timestamps.min(),
        "| max:", timestamps.max(),
        "| already sorted:", timestamps.is_monotonic_increasing,
    )

C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\512261312.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(


Train_Test_IoT_Fridge | parse failures: 0 | min: 2019-03-31 12:36:52 | max: 2019-04-29 01:13:08 | already sorted: False


C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\512261312.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(


Train_Test_IoT_Garage_Door | parse failures: 0 | min: 2019-03-31 12:41:32 | max: 2019-04-28 23:55:18 | already sorted: False


C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\512261312.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(


Train_Test_IoT_GPS_Tracker | parse failures: 0 | min: 2019-03-31 12:36:52 | max: 2019-04-28 11:44:04 | already sorted: False


... (4 outputs left out)

C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\512261312.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(


Train_Test_IoT_Thermostat | parse failures: 0 | min: 2019-03-31 12:43:33 | max: 2019-04-28 22:12:49 | already sorted: False


C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\512261312.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(


Train_Test_IoT_Weather | parse failures: 0 | min: 2019-03-31 12:40:22 | max: 2019-04-28 13:59:39 | already sorted: False


In [15]:
import pandas as pd

TARGET = "label"
datasets_sorted = {}
split_report = []

for name, df in datasets.items():
    df = df.copy()

    # format="mixed" suppresses the inference warning while parsing
    # each value according to its own format.
    df["_timestamp"] = pd.to_datetime(
        df["date"].astype(str) + " " + df["time"].astype(str),
        format="mixed",
        errors="coerce"
    )

    if df["_timestamp"].isna().any():
        print(f"WARNING: {name} has unparseable timestamps")
        continue

    df = df.sort_values("_timestamp").reset_index(drop=True)
    datasets_sorted[name] = df

    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    partitions = {
        "train": df.iloc[:train_end],
        "validation": df.iloc[train_end:val_end],
        "test": df.iloc[val_end:],
    }

    for split_name, split_df in partitions.items():
        counts = split_df[TARGET].value_counts().to_dict()
        split_report.append({
            "client": name,
            "split": split_name,
            "rows": len(split_df),
            "start": split_df["_timestamp"].min(),
            "end": split_df["_timestamp"].max(),
            "label_0": counts.get(0, 0),
            "label_1": counts.get(1, 0),
        })

split_report = pd.DataFrame(split_report)
display(split_report)

,client,split,rows,start,end,label_0,label_1
0,Train_Test_IoT_Fridge,train,27960,2019-03-31 12:36:52,2019-04-26 16:07:38,15000,12960
1,Train_Test_IoT_Fridge,validation,5992,2019-04-26 16:07:43,2019-04-28 05:02:29,0,5992
2,Train_Test_IoT_Fridge,test,5992,2019-04-28 05:02:30,2019-04-29 01:13:08,0,5992
3,Train_Test_IoT_Garage_Door,train,27710,2019-03-31 12:41:32,2019-04-26 14:48:06,15000,12710
4,Train_Test_IoT_Garage_Door,validation,5938,2019-04-26 14:48:06,2019-04-28 05:03:41,0,5938
5,Train_Test_IoT_Garage_Door,test,5939,2019-04-28 05:03:43,2019-04-28 23:55:18,0,5939
6,Train_Test_IoT_GPS_Tracker,train,27272,2019-03-31 12:36:52,2019-04-26 15:03:03,15000,12272
7,Train_Test_IoT_GPS_Tracker,validation,5844,2019-04-26 15:03:04,2019-04-28 05:06:00,0,5844
8,Train_Test_IoT_GPS_Tracker,test,5844,2019-04-28 05:06:02,2019-04-28 11:44:04,0,5844
9,Train_Test_IoT_Modbus,train,21774,2019-03-31 12:40:58,2019-04-26 14:40:18,15000,6774


In [16]:
from sklearn.model_selection import train_test_split
import pandas as pd

TARGET = "label"
RANDOM_STATE = 42

client_splits = {}
split_summary = []

for client_name, df in datasets.items():
    # Do not use target columns or raw date/time as model inputs.
    feature_cols = [
        col for col in df.columns
        if col.lower() not in {"label", "type", "date", "time"}
    ]

    X = df[feature_cols].copy()
    y = df[TARGET].astype(int).copy()

    # 70% train, 15% validation, 15% test.
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )

    client_splits[client_name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_cols": feature_cols,
    }

    for split_name, labels in [
        ("train", y_train),
        ("validation", y_val),
        ("test", y_test),
    ]:
        counts = labels.value_counts().to_dict()
        split_summary.append({
            "client": client_name,
            "split": split_name,
            "rows": len(labels),
            "label_0": counts.get(0, 0),
            "label_1": counts.get(1, 0),
        })

display(pd.DataFrame(split_summary))

,client,split,rows,label_0,label_1
0,Train_Test_IoT_Fridge,train,27960,10500,17460
1,Train_Test_IoT_Fridge,validation,5992,2250,3742
2,Train_Test_IoT_Fridge,test,5992,2250,3742
3,Train_Test_IoT_Garage_Door,train,27710,10500,17210
4,Train_Test_IoT_Garage_Door,validation,5938,2250,3688
5,Train_Test_IoT_Garage_Door,test,5939,2250,3689
6,Train_Test_IoT_GPS_Tracker,train,27272,10500,16772
7,Train_Test_IoT_GPS_Tracker,validation,5844,2250,3594
8,Train_Test_IoT_GPS_Tracker,test,5844,2250,3594
9,Train_Test_IoT_Modbus,train,21774,10500,11274


In [18]:
TARGET_COLUMNS = {"label", "type"}

duplicate_report = []

for client_name, df in datasets.items():
    # Exact duplicate rows, including labels and timestamps
    exact_duplicate_rows = int(df.duplicated().sum())

    # Same model inputs appearing with conflicting labels
    feature_cols = [
        col for col in df.columns
        if col.lower() not in {"label", "type", "date", "time"}
    ]

    if feature_cols:
        conflicting_label_groups = (
            df.groupby(feature_cols, dropna=False)["label"]
              .nunique()
              .gt(1)
              .sum()
        )
    else:
        conflicting_label_groups = 0

    duplicate_report.append({
        "client": client_name,
        "exact_duplicate_rows": exact_duplicate_rows,
        "feature_groups_with_conflicting_labels": int(conflicting_label_groups),
    })

display(pd.DataFrame(duplicate_report))

,client,exact_duplicate_rows,feature_groups_with_conflicting_labels
0,Train_Test_IoT_Fridge,2834,0
1,Train_Test_IoT_Garage_Door,20221,0
2,Train_Test_IoT_GPS_Tracker,0,1
3,Train_Test_IoT_Modbus,13314,0
4,Train_Test_IoT_Motion_Light,13658,2
5,Train_Test_IoT_Thermostat,424,2
6,Train_Test_IoT_Weather,0,0


In [19]:
from sklearn.model_selection import train_test_split
import pandas as pd

TARGET = "label"
RANDOM_STATE = 42

datasets_dedup = {}
client_splits = {}
dedup_report = []
split_summary = []

for client_name, original_df in datasets.items():
    # Remove rows identical across every column, including timestamp and labels.
    df = original_df.drop_duplicates().copy()

    datasets_dedup[client_name] = df

    dedup_report.append({
        "client": client_name,
        "original_rows": len(original_df),
        "rows_after_dedup": len(df),
        "exact_duplicates_removed": len(original_df) - len(df),
    })

    feature_cols = [
        col for col in df.columns
        if col.lower() not in {"label", "type", "date", "time"}
    ]

    X = df[feature_cols].copy()
    y = df[TARGET].astype(int).copy()

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )

    client_splits[client_name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_cols": feature_cols,
    }

    for split_name, labels in [
        ("train", y_train),
        ("validation", y_val),
        ("test", y_test),
    ]:
        counts = labels.value_counts().to_dict()
        split_summary.append({
            "client": client_name,
            "split": split_name,
            "rows": len(labels),
            "label_0": counts.get(0, 0),
            "label_1": counts.get(1, 0),
        })

display(pd.DataFrame(dedup_report))
display(pd.DataFrame(split_summary))

,client,original_rows,rows_after_dedup,exact_duplicates_removed
0,Train_Test_IoT_Fridge,39944,37110,2834
1,Train_Test_IoT_Garage_Door,39587,19366,20221
2,Train_Test_IoT_GPS_Tracker,38960,38960,0
3,Train_Test_IoT_Modbus,31106,17792,13314
4,Train_Test_IoT_Motion_Light,39488,25830,13658
5,Train_Test_IoT_Thermostat,32774,32350,424
6,Train_Test_IoT_Weather,39260,39260,0


,client,split,rows,label_0,label_1
0,Train_Test_IoT_Fridge,train,25977,8597,17380
1,Train_Test_IoT_Fridge,validation,5566,1842,3724
2,Train_Test_IoT_Fridge,test,5567,1843,3724
3,Train_Test_IoT_Garage_Door,train,13556,412,13144
4,Train_Test_IoT_Garage_Door,validation,2905,88,2817
5,Train_Test_IoT_Garage_Door,test,2905,89,2816
6,Train_Test_IoT_GPS_Tracker,train,27272,10500,16772
7,Train_Test_IoT_GPS_Tracker,validation,5844,2250,3594
8,Train_Test_IoT_GPS_Tracker,test,5844,2250,3594
9,Train_Test_IoT_Modbus,train,12454,2734,9720


In [20]:
dedup_class_report = []

for client_name, original_df in datasets.items():
    dedup_df = original_df.drop_duplicates()

    for label_value in sorted(original_df["label"].dropna().unique()):
        original_count = int((original_df["label"] == label_value).sum())
        dedup_count = int((dedup_df["label"] == label_value).sum())

        dedup_class_report.append({
            "client": client_name,
            "label": int(label_value),
            "original_rows": original_count,
            "after_dedup": dedup_count,
            "removed": original_count - dedup_count,
        })

display(pd.DataFrame(dedup_class_report))

,client,label,original_rows,after_dedup,removed
0,Train_Test_IoT_Fridge,0,15000,12282,2718
1,Train_Test_IoT_Fridge,1,24944,24828,116
2,Train_Test_IoT_Garage_Door,0,15000,589,14411
3,Train_Test_IoT_Garage_Door,1,24587,18777,5810
4,Train_Test_IoT_GPS_Tracker,0,15000,15000,0
5,Train_Test_IoT_GPS_Tracker,1,23960,23960,0
6,Train_Test_IoT_Modbus,0,15000,3906,11094
7,Train_Test_IoT_Modbus,1,16106,13886,2220
8,Train_Test_IoT_Motion_Light,0,15000,3449,11551
9,Train_Test_IoT_Motion_Light,1,24488,22381,2107


In [21]:
for client_name, df in datasets.items():
    duplicate_mask = df.duplicated(keep=False)
    duplicate_rows = df.loc[duplicate_mask]

    print(f"\n--- {client_name} ---")
    print("Rows belonging to duplicate groups:", len(duplicate_rows))
    if len(duplicate_rows):
        print("Duplicate rows by label:")
        display(duplicate_rows["label"].value_counts().sort_index().to_frame("rows"))
        print("Duplicate rows by type:")
        display(duplicate_rows["type"].value_counts().to_frame("rows"))


--- Train_Test_IoT_Fridge ---
Rows belonging to duplicate groups: 4916
Duplicate rows by label:


,rows
label,
0,4685
1,231


Duplicate rows by type:


,rows
type,
normal,4685
ddos,72
password,55
injection,50
xss,50
ransomware,4



--- Train_Test_IoT_Garage_Door ---
Rows belonging to duplicate groups: 25798
Duplicate rows by label:


,rows
label,
0,14928
1,10870


... (11 outputs left out)

,rows
label,
0,532
1,20


Duplicate rows by type:


,rows
type,
normal,532
injection,14
ransomware,6



--- Train_Test_IoT_Weather ---
Rows belonging to duplicate groups: 0


In [22]:
from sklearn.model_selection import train_test_split
import pandas as pd

client_splits_grouped = {}
split_report = []

for client_name, df in datasets.items():
    # Give identical full rows the same group ID.
    group_id = df.groupby(
        list(df.columns),
        dropna=False,
        sort=False
    ).ngroup()

    # Each exact-row group has one binary label.
    groups = pd.DataFrame({
        "group_id": group_id,
        "label": df["label"].to_numpy()
    }).drop_duplicates("group_id")

    # Split groups: 70% train, then divide the remaining 30% equally.
    train_groups, temp_groups = train_test_split(
        groups["group_id"],
        test_size=0.30,
        random_state=42,
        stratify=groups["label"]
    )

    temp_info = groups[groups["group_id"].isin(temp_groups)]
    val_groups, test_groups = train_test_split(
        temp_info["group_id"],
        test_size=0.50,
        random_state=42,
        stratify=temp_info["label"]
    )

    # Assign every row in a duplicate group to the same split.
    split_name = pd.Series(index=df.index, dtype="object")
    split_name.loc[group_id.isin(train_groups)] = "train"
    split_name.loc[group_id.isin(val_groups)] = "val"
    split_name.loc[group_id.isin(test_groups)] = "test"

    client_splits_grouped[client_name] = {
        split: df.loc[split_name == split].copy()
        for split in ["train", "val", "test"]
    }

    for split, split_df in client_splits_grouped[client_name].items():
        split_report.append({
            "client": client_name,
            "split": split,
            "rows": len(split_df),
            "normal": int((split_df["label"] == 0).sum()),
            "attack": int((split_df["label"] == 1).sum()),
            "attack_rate": round(float(split_df["label"].mean()), 4)
        })

display(pd.DataFrame(split_report))

,client,split,rows,normal,attack,attack_rate
0,Train_Test_IoT_Fridge,train,27891,10434,17457,0.6259
1,Train_Test_IoT_Fridge,val,6032,2289,3743,0.6205
2,Train_Test_IoT_Fridge,test,6021,2277,3744,0.6218
3,Train_Test_IoT_Garage_Door,train,28198,10983,17215,0.6105
4,Train_Test_IoT_Garage_Door,val,5680,1993,3687,0.6491
5,Train_Test_IoT_Garage_Door,test,5709,2024,3685,0.6455
6,Train_Test_IoT_GPS_Tracker,train,27272,10500,16772,0.6150
7,Train_Test_IoT_GPS_Tracker,val,5844,2250,3594,0.6150
8,Train_Test_IoT_GPS_Tracker,test,5844,2250,3594,0.6150
9,Train_Test_IoT_Modbus,train,21783,10512,11271,0.5174


In [23]:
for client_name, splits in client_splits_grouped.items():
    train_rows = set(map(tuple, splits["train"].itertuples(index=False, name=None)))
    val_rows   = set(map(tuple, splits["val"].itertuples(index=False, name=None)))
    test_rows  = set(map(tuple, splits["test"].itertuples(index=False, name=None)))

    assert not (train_rows & val_rows)
    assert not (train_rows & test_rows)
    assert not (val_rows & test_rows)

print("No exact full rows overlap across splits.")

No exact full rows overlap across splits.


In [24]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

client_preprocessors = {}
client_data = {}

def add_time_features(df):
    out = df.copy()

    # Combine the original date and time columns, then derive cyclical features.
    timestamp = pd.to_datetime(
        out["date"].astype(str) + " " + out["time"].astype(str),
        errors="coerce"
    )

    hour = timestamp.dt.hour + timestamp.dt.minute / 60
    weekday = timestamp.dt.dayofweek

    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    out["weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
    out["weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

    # Do not feed raw date/time or target columns to the model.
    out = out.drop(columns=["date", "time", "label", "type"], errors="ignore")
    return out

def dense_array(x):
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)

for client_name, splits in client_splits_grouped.items():
    X_train_df = add_time_features(splits["train"])
    X_val_df   = add_time_features(splits["val"])
    X_test_df  = add_time_features(splits["test"])

    y_train = splits["train"]["label"].astype("int64").to_numpy()
    y_val   = splits["val"]["label"].astype("int64").to_numpy()
    y_test  = splits["test"]["label"].astype("int64").to_numpy()

    categorical_cols = X_train_df.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_cols = [
        col for col in X_train_df.columns
        if col not in categorical_cols
    ]

    preprocess = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_cols),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ],
        remainder="drop"
    )

    # Fit only on train; transform validation and test with the same fitted pipeline.
    X_train = dense_array(preprocess.fit_transform(X_train_df))
    X_val   = dense_array(preprocess.transform(X_val_df))
    X_test  = dense_array(preprocess.transform(X_test_df))

    client_preprocessors[client_name] = preprocess
    client_data[client_name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": preprocess.get_feature_names_out()
    }

    print(
        client_name,
        "| train:", X_train.shape,
        "| val:", X_val.shape,
        "| test:", X_test.shape
    )

C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamp = pd.to_datetime(
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday = timestamp.dt.dayofweek
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamp = pd.to_datetime(
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday = t

Train_Test_IoT_Fridge | train: (27891, 11) | val: (6032, 11) | test: (6021, 11)


C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday = timestamp.dt.dayofweek
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamp = pd.to_datetime(
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday = timestamp.dt.dayofweek
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timest

Train_Test_IoT_Garage_Door | train: (28198, 10) | val: (5680, 10) | test: (5709, 10)


C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday = timestamp.dt.dayofweek
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamp = pd.to_datetime(
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday = timestamp.dt.dayofweek
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timest

Train_Test_IoT_GPS_Tracker | train: (27272, 6) | val: (5844, 6) | test: (5844, 6)


... (5 outputs left out)

Train_Test_IoT_Thermostat | train: (22947, 6) | val: (4926, 6) | test: (4901, 6)


C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday = timestamp.dt.dayofweek
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamp = pd.to_datetime(
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday = timestamp.dt.dayofweek
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timest

Train_Test_IoT_Weather | train: (27482, 7) | val: (5889, 7) | test: (5889, 7)


C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\3153755408.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekday = timestamp.dt.dayofweek


In [25]:
for client_name, splits in client_splits_grouped.items():
    for split_name, df in splits.items():
        timestamp = pd.to_datetime(
            df["date"].astype(str) + " " + df["time"].astype(str),
            errors="coerce"
        )
        missing = int(timestamp.isna().sum())
        if missing:
            print(client_name, split_name, "unparsed timestamps:", missing)

print("Timestamp check complete.")

C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\2401024899.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamp = pd.to_datetime(
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\2401024899.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamp = pd.to_datetime(
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\2401024899.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamp = pd.to_datetime(
C:\Users\aayus\AppData\Local\Temp\ipykernel_10376\2401024899.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling 

Timestamp check complete.


### Reference Baseline

In [26]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

local_models = {}
val_results = []

for client_name, data in client_data.items():
    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(data["X_train"], data["y_train"])
    local_models[client_name] = model

    y_true = data["y_val"]
    y_pred = model.predict(data["X_val"])
    y_prob = model.predict_proba(data["X_val"])[:, 1]

    val_results.append({
        "client": client_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    })

val_results_df = pd.DataFrame(val_results)
display(val_results_df.round(4))

,client,accuracy,precision,recall,f1,roc_auc
0,Train_Test_IoT_Fridge,1.0000,1.0,1.0000,1.0000,1.0000
1,Train_Test_IoT_Garage_Door,1.0000,1.0,1.0000,1.0000,1.0000
2,Train_Test_IoT_GPS_Tracker,1.0000,1.0,1.0000,1.0000,1.0000
3,Train_Test_IoT_Modbus,0.9858,1.0,0.9727,0.9862,0.9999
4,Train_Test_IoT_Motion_Light,0.9966,1.0,0.9946,0.9973,1.0000
5,Train_Test_IoT_Thermostat,0.9982,1.0,0.9966,0.9983,0.9971
6,Train_Test_IoT_Weather,0.9642,1.0,0.9420,0.9701,0.9949


In [27]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import pandas as pd

test_results = []

for client_name, data in client_data.items():
    model = local_models[client_name]

    y_true = data["y_test"]
    y_pred = model.predict(data["X_test"])
    y_prob = model.predict_proba(data["X_test"])[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    test_results.append({
        "client": client_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

test_results_df = pd.DataFrame(test_results)
display(test_results_df.round(4))

,client,accuracy,precision,recall,f1,roc_auc,TN,FP,FN,TP
0,Train_Test_IoT_Fridge,1.0000,1.0,1.0000,1.0000,1.0000,2277,0,0,3744
1,Train_Test_IoT_Garage_Door,1.0000,1.0,1.0000,1.0000,1.0000,2024,0,0,3685
2,Train_Test_IoT_GPS_Tracker,1.0000,1.0,1.0000,1.0000,1.0000,2250,0,0,3594
3,Train_Test_IoT_Modbus,0.9871,1.0,0.9752,0.9874,0.9999,2246,0,60,2356
4,Train_Test_IoT_Motion_Light,0.9954,1.0,0.9924,0.9962,1.0000,2418,0,28,3664
5,Train_Test_IoT_Thermostat,0.9986,1.0,0.9974,0.9987,0.9977,2234,0,7,2660
6,Train_Test_IoT_Weather,0.9584,1.0,0.9327,0.9652,0.9932,2250,0,245,3394


In [28]:
import copy
import torch
import torch.nn as nn
import numpy as np

# Use the CPU for this first version
device = torch.device("cpu")

# Each client has its own input layer, because feature counts differ.
# The later layers have the same shape and can be shared.
class PersonalizedNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.personal_input = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU()
        )

        self.shared_layers = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.personal_input(x)
        return self.shared_layers(x).squeeze(1)


# Create one model for each device, using its own feature count.
client_models = {}

for client, data in client_data.items():
    input_size = data["X_train"].shape[1]
    client_models[client] = PersonalizedNet(input_size).to(device)

print("Created models for", len(client_models), "clients.")

for client, model in client_models.items():
    print(client, "input features:", model.personal_input[0].in_features)

Created models for 7 clients.
Train_Test_IoT_Fridge input features: 11
Train_Test_IoT_Garage_Door input features: 10
Train_Test_IoT_GPS_Tracker input features: 6
Train_Test_IoT_Modbus input features: 8
Train_Test_IoT_Motion_Light input features: 7
Train_Test_IoT_Thermostat input features: 6
Train_Test_IoT_Weather input features: 7


In [29]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Start with the shared layers from one of the client models
global_shared = copy.deepcopy(
    next(iter(client_models.values())).shared_layers.state_dict()
)

rounds = 5
local_epochs = 1
batch_size = 256
learning_rate = 0.001

for round_num in range(rounds):
    client_updates = []
    client_sizes = []

    for client, model in client_models.items():
        data = client_data[client]

        # Give this client the latest shared layers.
        # Its personal_input layer is not replaced.
        model.shared_layers.load_state_dict(global_shared)

        X = torch.tensor(
            np.asarray(data["X_train"], dtype=np.float32)
        )
        y = torch.tensor(
            np.asarray(data["y_train"], dtype=np.float32).reshape(-1)
        )

        loader = DataLoader(
            TensorDataset(X, y),
            batch_size=batch_size,
            shuffle=True
        )

        # Account for the client's class imbalance during training.
        positives = y.sum().item()
        negatives = len(y) - positives
        pos_weight = torch.tensor(
            [negatives / max(positives, 1)],
            dtype=torch.float32
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

        model.train()
        for _ in range(local_epochs):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                logits = model(batch_X)
                loss = criterion(logits, batch_y)
                loss.backward()
                optimizer.step()

        # Save this client's updated shared layers.
        client_updates.append(
            copy.deepcopy(model.shared_layers.state_dict())
        )
        client_sizes.append(len(y))

    # Average shared layers, weighted by each client's training-set size.
    total_size = sum(client_sizes)
    new_global_shared = {}

    for key in global_shared:
        new_global_shared[key] = sum(
            client_updates[i][key] * client_sizes[i]
            for i in range(len(client_updates))
        ) / total_size

    global_shared = new_global_shared

    print(f"Completed federated round {round_num + 1}/{rounds}")

# Make sure every client model ends with the final shared layers.
for model in client_models.values():
    model.shared_layers.load_state_dict(global_shared)

print("Federated training complete.")

Completed federated round 1/5
Completed federated round 2/5
Completed federated round 3/5
Completed federated round 4/5
Completed federated round 5/5
Federated training complete.


In [30]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

federated_val_results = []

for client, model in client_models.items():
    data = client_data[client]

    X_val = torch.tensor(
        np.asarray(data["X_val"], dtype=np.float32)
    )
    y_val = np.asarray(data["y_val"], dtype=int).reshape(-1)

    model.eval()
    with torch.no_grad():
        logits = model(X_val)
        probabilities = torch.sigmoid(logits).cpu().numpy()

    predictions = (probabilities >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    federated_val_results.append({
        "client": client,
        "accuracy": accuracy_score(y_val, predictions),
        "precision": precision_score(y_val, predictions, zero_division=0),
        "recall": recall_score(y_val, predictions, zero_division=0),
        "f1": f1_score(y_val, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_val, probabilities),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

federated_val_results_df = pd.DataFrame(federated_val_results)

display(
    federated_val_results_df.round(4)
)

,client,accuracy,precision,recall,f1,roc_auc,TN,FP,FN,TP
0,Train_Test_IoT_Fridge,1.0000,1.0000,1.0000,1.0000,1.0000,2289,0,0,3743
1,Train_Test_IoT_Garage_Door,1.0000,1.0000,1.0000,1.0000,1.0000,1993,0,0,3687
2,Train_Test_IoT_GPS_Tracker,1.0000,1.0000,1.0000,1.0000,1.0000,2250,0,0,3594
3,Train_Test_IoT_Modbus,0.9612,1.0000,0.9252,0.9611,0.9948,2242,0,181,2238
4,Train_Test_IoT_Motion_Light,0.9998,1.0000,0.9997,0.9999,1.0000,2254,0,1,3678
5,Train_Test_IoT_Thermostat,0.9913,0.9874,0.9966,0.9920,0.9970,2226,34,9,2657
6,Train_Test_IoT_Weather,0.9812,1.0000,0.9695,0.9845,0.9976,2250,0,111,3528


In [31]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Start fresh: create new models for all 7 clients
torch.manual_seed(42)

client_models_10 = {
    client: PersonalizedNet(data["X_train"].shape[1])
    for client, data in client_data.items()
}

# Initialize the shared layers from the new models
global_shared_10 = copy.deepcopy(
    next(iter(client_models_10.values())).shared_layers.state_dict()
)

rounds = 10
local_epochs = 1
batch_size = 256
learning_rate = 0.001

for round_num in range(rounds):
    client_updates = []
    client_sizes = []

    for client, model in client_models_10.items():
        data = client_data[client]

        # Apply the latest shared layers; keep this client's input layer
        model.shared_layers.load_state_dict(global_shared_10)

        X = torch.tensor(np.asarray(data["X_train"], dtype=np.float32))
        y = torch.tensor(
            np.asarray(data["y_train"], dtype=np.float32).reshape(-1)
        )

        loader = DataLoader(
            TensorDataset(X, y),
            batch_size=batch_size,
            shuffle=True
        )

        positives = y.sum().item()
        negatives = len(y) - positives
        pos_weight = torch.tensor(
            [negatives / max(positives, 1)],
            dtype=torch.float32
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

        model.train()
        for _ in range(local_epochs):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                loss = criterion(model(batch_X), batch_y)
                loss.backward()
                optimizer.step()

        client_updates.append(
            copy.deepcopy(model.shared_layers.state_dict())
        )
        client_sizes.append(len(y))

    # Average the shared layers, weighted by each client's training size
    total_size = sum(client_sizes)
    global_shared_10 = {
        key: sum(
            client_updates[i][key] * client_sizes[i]
            for i in range(len(client_updates))
        ) / total_size
        for key in global_shared_10
    }

    print(f"Completed round {round_num + 1}/{rounds}")

# Put the final shared layers into every client model
for model in client_models_10.values():
    model.shared_layers.load_state_dict(global_shared_10)

print("10-round federated training complete.")

Completed round 1/10
Completed round 2/10
Completed round 3/10
Completed round 4/10
Completed round 5/10
Completed round 6/10
Completed round 7/10
Completed round 8/10
Completed round 9/10
Completed round 10/10
10-round federated training complete.


In [32]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

federated_10_val_results = []

for client, model in client_models_10.items():
    data = client_data[client]

    X_val = torch.tensor(
        np.asarray(data["X_val"], dtype=np.float32)
    )
    y_val = np.asarray(data["y_val"], dtype=int).reshape(-1)

    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(X_val)).cpu().numpy()

    predictions = (probabilities >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    federated_10_val_results.append({
        "client": client,
        "accuracy": accuracy_score(y_val, predictions),
        "precision": precision_score(y_val, predictions, zero_division=0),
        "recall": recall_score(y_val, predictions, zero_division=0),
        "f1": f1_score(y_val, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_val, probabilities),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

federated_10_val_results_df = pd.DataFrame(federated_10_val_results)
display(federated_10_val_results_df.round(4))

,client,accuracy,precision,recall,f1,roc_auc,TN,FP,FN,TP
0,Train_Test_IoT_Fridge,1.0000,1.0,1.0000,1.0000,1.0000,2289,0,0,3743
1,Train_Test_IoT_Garage_Door,1.0000,1.0,1.0000,1.0000,1.0000,1993,0,0,3687
2,Train_Test_IoT_GPS_Tracker,1.0000,1.0,1.0000,1.0000,1.0000,2250,0,0,3594
3,Train_Test_IoT_Modbus,0.9933,1.0,0.9872,0.9936,0.9994,2242,0,31,2388
4,Train_Test_IoT_Motion_Light,1.0000,1.0,1.0000,1.0000,1.0000,2254,0,0,3679
5,Train_Test_IoT_Thermostat,0.9994,1.0,0.9989,0.9994,1.0000,2260,0,3,2663
6,Train_Test_IoT_Weather,0.9902,1.0,0.9841,0.9920,0.9986,2250,0,58,3581


In [33]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

federated_10_test_results = []

for client, model in client_models_10.items():
    data = client_data[client]

    X_test = torch.tensor(
        np.asarray(data["X_test"], dtype=np.float32)
    )
    y_test = np.asarray(data["y_test"], dtype=int).reshape(-1)

    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(X_test)).cpu().numpy()

    predictions = (probabilities >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    federated_10_test_results.append({
        "client": client,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

federated_10_test_results_df = pd.DataFrame(federated_10_test_results)
display(federated_10_test_results_df.round(4))

,client,accuracy,precision,recall,f1,roc_auc,TN,FP,FN,TP
0,Train_Test_IoT_Fridge,1.0000,1.0,1.0000,1.0000,1.0000,2277,0,0,3744
1,Train_Test_IoT_Garage_Door,1.0000,1.0,1.0000,1.0000,1.0000,2024,0,0,3685
2,Train_Test_IoT_GPS_Tracker,1.0000,1.0,1.0000,1.0000,1.0000,2250,0,0,3594
3,Train_Test_IoT_Modbus,0.9910,1.0,0.9826,0.9912,0.9992,2246,0,42,2374
4,Train_Test_IoT_Motion_Light,1.0000,1.0,1.0000,1.0000,1.0000,2418,0,0,3692
5,Train_Test_IoT_Thermostat,0.9998,1.0,0.9996,0.9998,1.0000,2234,0,1,2666
6,Train_Test_IoT_Weather,0.9903,1.0,0.9843,0.9921,0.9990,2250,0,57,3582


In [37]:
for client, preprocessor in client_preprocessors.items():
    try:
        names = preprocessor.get_feature_names_out()
        print(f"\n{client} ({len(names)} features)")
        print(list(names))
    except Exception as e:
        print(f"\n{client}: couldn't get feature names — {e}")


Train_Test_IoT_Fridge (11 features)
['numeric__fridge_temperature', 'numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric__weekday_cos', 'categorical__temp_condition_high', 'categorical__temp_condition_high ', 'categorical__temp_condition_high  ', 'categorical__temp_condition_low', 'categorical__temp_condition_low ', 'categorical__temp_condition_low  ']

Train_Test_IoT_Garage_Door (10 features)
['numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric__weekday_cos', 'categorical__door_state_closed', 'categorical__door_state_open', 'categorical__sphone_signal_0', 'categorical__sphone_signal_1', 'categorical__sphone_signal_false  ', 'categorical__sphone_signal_true  ']

Train_Test_IoT_GPS_Tracker (6 features)
['numeric__latitude', 'numeric__longitude', 'numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric__weekday_cos']

Train_Test_IoT_Modbus (8 features)
['numeric__FC1_Read_Input_Register', 'numeric__FC2_Read_Discrete_Value', 

In [38]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

single_feature_results = []

for client, data in client_data.items():
    feature_names = client_preprocessors[client].get_feature_names_out()

    X_train = np.asarray(data["X_train"])
    X_val = np.asarray(data["X_val"])
    y_train = np.asarray(data["y_train"]).reshape(-1)
    y_val = np.asarray(data["y_val"]).reshape(-1)

    for i, feature_name in enumerate(feature_names):
        # A shallow tree tests how much this one feature can tell us.
        model = DecisionTreeClassifier(max_depth=1, random_state=42)
        model.fit(X_train[:, [i]], y_train)

        predictions = model.predict(X_val[:, [i]])

        single_feature_results.append({
            "client": client,
            "feature": feature_name,
            "val_accuracy": accuracy_score(y_val, predictions),
            "val_f1": f1_score(y_val, predictions, zero_division=0)
        })

single_feature_df = pd.DataFrame(single_feature_results)

display(
    single_feature_df
    .sort_values(["client", "val_f1"], ascending=[True, False])
    .round(4)
)

,client,feature,val_accuracy,val_f1
2,Train_Test_IoT_Fridge,numeric__hour_cos,0.8422,0.8865
6,Train_Test_IoT_Fridge,categorical__temp_condition_high,0.7722,0.8449
9,Train_Test_IoT_Fridge,categorical__temp_condition_low,0.7377,0.8255
3,Train_Test_IoT_Fridge,numeric__weekday_sin,0.7311,0.8219
4,Train_Test_IoT_Fridge,numeric__weekday_cos,0.8016,0.8097
7,Train_Test_IoT_Fridge,categorical__temp_condition_high,0.6787,0.7944
10,Train_Test_IoT_Fridge,categorical__temp_condition_low,0.6729,0.7914
0,Train_Test_IoT_Fridge,numeric__fridge_temperature,0.6205,0.7658
1,Train_Test_IoT_Fridge,numeric__hour_sin,0.6205,0.7658
5,Train_Test_IoT_Fridge,categorical__temp_condition_high,0.7352,0.7288


In [39]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score

time_features = {"hour_sin", "hour_cos", "weekday_sin", "weekday_cos"}
results_without_time = []

for client, data in client_data.items():
    feature_names = client_preprocessors[client].get_feature_names_out()

    # Keep every feature except the four time features
    keep_indices = [
        i for i, name in enumerate(feature_names)
        if name.split("__")[-1].strip() not in time_features
    ]

    X_train = np.asarray(data["X_train"])[:, keep_indices]
    X_val = np.asarray(data["X_val"])[:, keep_indices]
    y_train = np.asarray(data["y_train"]).reshape(-1)
    y_val = np.asarray(data["y_val"]).reshape(-1)

    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    results_without_time.append({
        "client": client,
        "f1_without_time": f1_score(y_val, predictions, zero_division=0),
        "recall_without_time": recall_score(y_val, predictions, zero_division=0),
        "accuracy_without_time": accuracy_score(y_val, predictions)
    })

results_without_time_df = pd.DataFrame(results_without_time)
display(results_without_time_df.round(4))

,client,f1_without_time,recall_without_time,accuracy_without_time
0,Train_Test_IoT_Fridge,1.0000,1.0000,1.0000
1,Train_Test_IoT_Garage_Door,1.0000,1.0000,1.0000
2,Train_Test_IoT_GPS_Tracker,0.7513,0.6711,0.7267
3,Train_Test_IoT_Modbus,0.5212,0.5126,0.5113
4,Train_Test_IoT_Motion_Light,0.6184,0.6235,0.5228
5,Train_Test_IoT_Thermostat,0.6801,0.9002,0.5416
6,Train_Test_IoT_Weather,0.5999,0.5529,0.5442


In [40]:
datasets["Train_Test_IoT_Weather"].columns

Index(['date', 'time', 'temperature', 'pressure', 'humidity', 'label', 'type'],
      dtype='str')

In [41]:
weather = datasets["Train_Test_IoT_Weather"]

print("Date examples:")
print(weather["date"].head(10).to_list())

print("\nTime examples:")
print(weather["time"].head(10).to_list())

print("\nColumn types:")
print(weather[["date", "time"]].dtypes)

Date examples:
['25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19', '25-Apr-19']

Time examples:
['17:33:16', '17:33:16', '17:33:16', '17:33:21', '17:33:21', '17:33:21', '17:33:31', '17:33:31', '17:33:31', '17:33:36']

Column types:
date    str
time    str
dtype: object


In [42]:
import pandas as pd

weather = datasets["Train_Test_IoT_Weather"].copy()

weather["timestamp"] = pd.to_datetime(
    weather["date"] + " " + weather["time"],
    format="%d-%b-%y %H:%M:%S",
    errors="coerce"
)

print("Unparsed timestamps:", weather["timestamp"].isna().sum())
print("Earliest:", weather["timestamp"].min())
print("Latest:", weather["timestamp"].max())

print("\nFirst few rows in time order:")
print(
    weather.sort_values("timestamp")[["date", "time", "timestamp", "label"]]
    .head()
)

Unparsed timestamps: 11395
Earliest: 2019-03-31 12:40:22
Latest: 2019-04-28 05:27:24

First few rows in time order:
            date      time           timestamp  label
18116  31-Mar-19  12:40:22 2019-03-31 12:40:22      0
18127  31-Mar-19  12:40:22 2019-03-31 12:40:22      0
18128  31-Mar-19  12:40:22 2019-03-31 12:40:22      0
18129  31-Mar-19  12:40:22 2019-03-31 12:40:22      0
18130  31-Mar-19  12:40:22 2019-03-31 12:40:22      0


In [43]:
bad = weather[weather["timestamp"].isna()]

print("Failed rows:", len(bad))

print("\nDate values among failed rows:")
print(bad["date"].value_counts().head(20))

print("\nTime values among failed rows:")
print(bad["time"].value_counts().head(20))

print("\nExample failed rows:")
print(bad[["date", "time"]].head(15).to_string(index=False))

Failed rows: 11395

Date values among failed rows:
date
28-Apr-19    5000
26-Apr-19    5000
27-Apr-19     866
23-Apr-19     529
Name: count, dtype: int64

Time values among failed rows:
time
20:32:44     4
20:32:54     4
20:32:59     4
13:40:44     3
13:40:49     3
13:40:54     3
13:41:04     3
13:41:09     3
13:41:19     3
13:41:29     3
13:41:39     3
13:41:44     3
13:41:49     3
13:42:04     3
13:42:09     3
13:42:19     3
13:42:29     3
13:42:39     3
13:42:44     3
13:42:54     3
Name: count, dtype: int64

Example failed rows:
     date       time
28-Apr-19  08:02:06 
28-Apr-19  08:02:08 
28-Apr-19  08:02:09 
28-Apr-19  08:02:11 
28-Apr-19  08:02:13 
28-Apr-19  08:02:14 
28-Apr-19  08:02:16 
28-Apr-19  08:02:18 
28-Apr-19  08:02:19 
28-Apr-19  08:02:21 
28-Apr-19  08:02:23 
28-Apr-19  08:02:24 
28-Apr-19  08:02:26 
28-Apr-19  08:02:28 
28-Apr-19  08:02:29 


In [44]:
for col in ["date", "time"]:
    values = weather[col].astype("string").str.strip()

    print(f"\n{col}")
    print("Missing values:", weather[col].isna().sum())
    print("Blank/whitespace values:", values.eq("").sum())
    print("Examples from failed rows:")
    print([repr(x) for x in bad[col].head(10).tolist()])


date
Missing values: 0
Blank/whitespace values: 0
Examples from failed rows:
["'28-Apr-19'", "'28-Apr-19'", "'28-Apr-19'", "'28-Apr-19'", "'28-Apr-19'", "'28-Apr-19'", "'28-Apr-19'", "'28-Apr-19'", "'28-Apr-19'", "'28-Apr-19'"]

time
Missing values: 0
Blank/whitespace values: 0
Examples from failed rows:
["' 08:02:06 '", "' 08:02:08 '", "' 08:02:09 '", "' 08:02:11 '", "' 08:02:13 '", "' 08:02:14 '", "' 08:02:16 '", "' 08:02:18 '", "' 08:02:19 '", "' 08:02:21 '"]


In [45]:
weather = datasets["Train_Test_IoT_Weather"].copy()

date_clean = weather["date"].astype("string").str.strip()
time_clean = weather["time"].astype("string").str.strip()

weather["timestamp"] = pd.to_datetime(
    date_clean + " " + time_clean,
    format="%d-%b-%y %H:%M:%S",
    errors="coerce"
)

print("Unparsed timestamps:", weather["timestamp"].isna().sum())
print("Earliest:", weather["timestamp"].min())
print("Latest:", weather["timestamp"].max())

Unparsed timestamps: 0
Earliest: 2019-03-31 12:40:22
Latest: 2019-04-28 13:59:39


In [46]:
# Sort by timestamp; keep rows with identical timestamps together
weather_sorted = weather.sort_values("timestamp").copy()

unique_times = weather_sorted["timestamp"].drop_duplicates().sort_values()
n_times = len(unique_times)

train_end = unique_times.iloc[int(n_times * 0.70) - 1]
val_end = unique_times.iloc[int(n_times * 0.85) - 1]

weather_sorted["time_split"] = "test"
weather_sorted.loc[weather_sorted["timestamp"] <= train_end, "time_split"] = "train"
weather_sorted.loc[
    (weather_sorted["timestamp"] > train_end) &
    (weather_sorted["timestamp"] <= val_end),
    "time_split"
] = "validation"

print("Train ends:", train_end)
print("Validation ends:", val_end)

print("\nSplit sizes:")
print(weather_sorted["time_split"].value_counts())

print("\nAttack counts by split:")
print(pd.crosstab(weather_sorted["time_split"], weather_sorted["label"]))

print("\nAttack rate by split:")
print(weather_sorted.groupby("time_split")["label"].mean())

Train ends: 2019-04-28 04:38:13
Validation ends: 2019-04-28 08:27:29

Split sizes:
time_split
train         32351
validation     3471
test           3438
Name: count, dtype: int64

Attack counts by split:
label           0      1
time_split              
test            0   3438
train       15000  17351
validation      0   3471

Attack rate by split:
time_split
test          1.000000
train         0.536336
validation    1.000000
Name: label, dtype: float64


In [47]:
weather_sorted["date_only"] = weather_sorted["timestamp"].dt.strftime("%Y-%m-%d")

daily = (
    weather_sorted.groupby("date_only")["label"]
    .agg(rows="size", attacks="sum", attack_rate="mean")
)

print(daily.to_string())

             rows  attacks  attack_rate
date_only                              
2019-03-31  15000        0          0.0
2019-04-23    529      529          1.0
2019-04-25  10000    10000          1.0
2019-04-26   5000     5000          1.0
2019-04-27    866      866          1.0
2019-04-28   7865     7865          1.0


In [48]:
for client, df in datasets.items():
    temp = df.copy()

    date_clean = temp["date"].astype("string").str.strip()
    time_clean = temp["time"].astype("string").str.strip()

    temp["timestamp"] = pd.to_datetime(
        date_clean + " " + time_clean,
        format="%d-%b-%y %H:%M:%S",
        errors="coerce"
    )
    temp["date_only"] = temp["timestamp"].dt.strftime("%Y-%m-%d")

    daily = temp.groupby("date_only")["label"].agg(
        rows="size",
        attacks="sum",
        attack_rate="mean"
    )

    print(f"\n--- {client} ---")
    print(daily.to_string())


--- Train_Test_IoT_Fridge ---
             rows  attacks  attack_rate
date_only                              
2019-03-31   8942        0          0.0
2019-04-01   1794        0          0.0
2019-04-02   4264        0          0.0
2019-04-25  10000    10000          1.0
2019-04-26   5000     5000          1.0
2019-04-27   2042     2042          1.0
2019-04-28   6988     6988          1.0
2019-04-29    914      914          1.0

--- Train_Test_IoT_Garage_Door ---
             rows  attacks  attack_rate
date_only                              
2019-03-31   6604        0          0.0
2019-04-01    372        0          0.0
2019-04-02   8024        0          0.0
2019-04-23    529      529          1.0
2019-04-25  10000    10000          1.0
2019-04-26   5000     5000          1.0
2019-04-27   1156     1156          1.0
2019-04-28   7902     7902          1.0

--- Train_Test_IoT_GPS_Tracker ---
             rows  attacks  attack_rate
date_only                              
2019-03-31  11222

In [49]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, accuracy_score

time_features = {"hour_sin", "hour_cos", "weekday_sin", "weekday_cos"}
comparison = []

for client, data in client_data.items():
    # Feature names produced by this client's fitted preprocessor
    feature_names = client_preprocessors[client].get_feature_names_out()
    keep_idx = [
        i for i, name in enumerate(feature_names)
        if name.strip() not in time_features
    ]

    X_train = data["X_train"]
    X_val = data["X_val"]
    y_train = data["y_train"]
    y_val = data["y_val"]

    # Original local model, using all features
    original_model = local_models[client]
    original_pred = original_model.predict(X_val)

    # Ablation model, using only non-time features
    sensor_model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    )
    sensor_model.fit(X_train[:, keep_idx], y_train)
    sensor_pred = sensor_model.predict(X_val[:, keep_idx])

    comparison.append({
        "client": client,
        "f1_all_features": f1_score(y_val, original_pred),
        "f1_without_time": f1_score(y_val, sensor_pred),
        "recall_all_features": recall_score(y_val, original_pred),
        "recall_without_time": recall_score(y_val, sensor_pred),
        "accuracy_all_features": accuracy_score(y_val, original_pred),
        "accuracy_without_time": accuracy_score(y_val, sensor_pred),
    })

comparison_df = pd.DataFrame(comparison)
comparison_df.round(4)

,client,f1_all_features,f1_without_time,recall_all_features,recall_without_time,accuracy_all_features,accuracy_without_time
0,Train_Test_IoT_Fridge,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
1,Train_Test_IoT_Garage_Door,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,Train_Test_IoT_GPS_Tracker,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
3,Train_Test_IoT_Modbus,0.9862,0.9862,0.9727,0.9727,0.9858,0.9858
4,Train_Test_IoT_Motion_Light,0.9973,0.9973,0.9946,0.9946,0.9966,0.9966
5,Train_Test_IoT_Thermostat,0.9983,0.9983,0.9966,0.9966,0.9982,0.9982
6,Train_Test_IoT_Weather,0.9701,0.9701,0.9420,0.9420,0.9642,0.9642


In [50]:
for client in client_data:
    names = list(client_preprocessors[client].get_feature_names_out())

    time_cols = [
        name for name in names
        if any(part in name.lower() for part in ["hour", "weekday"])
    ]

    print(f"\n{client}")
    print("Time-related feature names:", time_cols)


Train_Test_IoT_Fridge
Time-related feature names: ['numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric__weekday_cos']

Train_Test_IoT_Garage_Door
Time-related feature names: ['numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric__weekday_cos']

Train_Test_IoT_GPS_Tracker
Time-related feature names: ['numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric__weekday_cos']

Train_Test_IoT_Modbus
Time-related feature names: ['numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric__weekday_cos']

Train_Test_IoT_Motion_Light
Time-related feature names: ['numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric__weekday_cos']

Train_Test_IoT_Thermostat
Time-related feature names: ['numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric__weekday_cos']

Train_Test_IoT_Weather
Time-related feature names: ['numeric__hour_sin', 'numeric__hour_cos', 'numeric__weekday_sin', 'numeric_

In [51]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, accuracy_score
import pandas as pd

time_features = {"hour_sin", "hour_cos", "weekday_sin", "weekday_cos"}
comparison = []

for client, data in client_data.items():
    feature_names = client_preprocessors[client].get_feature_names_out()

    # Handle names such as "numeric__hour_sin"
    keep_idx = [
        i for i, name in enumerate(feature_names)
        if name.split("__")[-1].strip() not in time_features
    ]

    X_train = data["X_train"]
    X_val = data["X_val"]
    y_train = data["y_train"]
    y_val = data["y_val"]

    sensor_model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    )
    sensor_model.fit(X_train[:, keep_idx], y_train)
    sensor_pred = sensor_model.predict(X_val[:, keep_idx])

    original_pred = local_models[client].predict(X_val)

    comparison.append({
        "client": client,
        "time_features_removed": len(feature_names) - len(keep_idx),
        "f1_all_features": f1_score(y_val, original_pred),
        "f1_without_time": f1_score(y_val, sensor_pred),
        "recall_all_features": recall_score(y_val, original_pred),
        "recall_without_time": recall_score(y_val, sensor_pred),
        "accuracy_all_features": accuracy_score(y_val, original_pred),
        "accuracy_without_time": accuracy_score(y_val, sensor_pred),
    })

comparison_df = pd.DataFrame(comparison)
comparison_df.round(4)

,client,time_features_removed,f1_all_features,f1_without_time,recall_all_features,recall_without_time,accuracy_all_features,accuracy_without_time
0,Train_Test_IoT_Fridge,4,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
1,Train_Test_IoT_Garage_Door,4,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,Train_Test_IoT_GPS_Tracker,4,1.0000,0.7513,1.0000,0.6711,1.0000,0.7267
3,Train_Test_IoT_Modbus,4,0.9862,0.5212,0.9727,0.5126,0.9858,0.5113
4,Train_Test_IoT_Motion_Light,4,0.9973,0.6184,0.9946,0.6235,0.9966,0.5228
5,Train_Test_IoT_Thermostat,4,0.9983,0.6801,0.9966,0.9002,0.9982,0.5416
6,Train_Test_IoT_Weather,4,0.9701,0.5999,0.9420,0.5529,0.9642,0.5442
